<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Lab - Building Agents with DSPy on Databricks
#### Duration: 30 minutes
## Overview
In this lab, you will build a tool-calling agent with **DSPy**, a framework where you declare *what* a language model should do with **signatures** and let **modules** decide *how* to prompt it. You will connect DSPy to a Databricks model, prototype one tool as a plain Python function, wrap a Unity Catalog function as a second tool, and combine them in a `dspy.ReAct` agent that you trace with MLflow. This notebook acts as a primer for the DSPy framework and is by no means comprehensive. 

Each step has a **TODO** with `<FILL_IN>` placeholders. The instructions above each TODO tell you what to fill in, and you can reveal the answer if you get stuck.

## Learning Objectives
By the end of this lab, you will be able to:
1. Configure DSPy to use a Databricks model serving endpoint
2. Define DSPy signatures, in shorthand and as classes
3. Prototype agent tools as plain Python functions, and wrap a Unity Catalog function as a DSPy tool
4. Build a `dspy.ReAct` agent that reasons and calls tools
5. Inspect an agent's trajectory and its MLflow trace

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 6</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V6</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>


## A. Classroom Setup

Run the next cell to install the libraries for this lab, set up your catalog and schema, load the Airbnb dataset, and register the `avg_neigh_price` UC function. It also creates these variables: `catalog_name`, `schema_name`, `username`, and `mlflow_experiment_path`.

In [0]:
%run ./Includes/Classroom-Setup-8

Installing 7 package(s) with uv...
  pyyaml==6.0.3
  databricks-sdk==0.145.0
  databricks-openai==0.17.1
  databricks-agents==1.12.0
  mlflow-skinny==3.16.1
  dspy==3.4.0
  pydantic==2.13.5
Library installation complete.


Starting Demo Environment Setup
Using config: Includes/config/config-8.yaml

[0] Compute check passed: Serverless 6 detected.

[1] Setting up catalog...
Vocareum workspace detected.
  Catalog 'labuser17579966_1791942688' found. Using it.

[2] Setting up schema...
  Catalog : labuser17579966_1791942688
  Schema  : agent_training

[3] Setting up Unity Gateway model service...
  Model service already exists and matches: labuser17579966_1791942688.agent_training.agent-training-model → databricks-gpt-5-2

[4] Processing datasets...
  Using Vocareum Delta Share: airbnb

[5] Registering 1 SQL UC tool(s)...

Creating Tools
  UC function created: avg_neigh_price (from sql/)

[6] Setting up MLflow...
  Setting tracking URI: databricks
Setting up experiment: dspy-agent-training
  Created experiment: /Workspace/Users/labuser17579966_1791942688@vocareum.com/dspy-agent-training
  Active experiment: /Workspace/Users/labuser17579966_1791942688@vocareum.com/dspy-agent-training
Refreshing environment

D

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


Catalog,"labuser17579966_1791942688<button onclick=""var t=document.createElement('textarea');t.value='labuser17579966_1791942688';document.body.appendChild(t);t.select();try{{document.execCommand('copy');}}catch(e){{}}document.body.removeChild(t);var b=this;var o=b.innerText;b.innerText='Copied';setTimeout(function(){{b.innerText=o;}},1200);"" title=""Copy"" style=""flex:0 0 auto;padding:2px 9px;font-size:11px;font-family:inherit;color:#1B5162;background:#F9F7F4;border:1px solid #EEEDE9;border-radius:5px;cursor:pointer"">Copy"
Schema,agent_trainingCopy
Table,"labuser17579966_1791942688.agent_training.sf_airbnb_listings<button onclick=""var t=document.createElement('textarea');t.value='labuser17579966_1791942688.agent_training.sf_airbnb_listings';document.body.appendChild(t);t.select();try{{document.execCommand('copy');}}catch(e){{}}document.body.removeChild(t);var b=this;var o=b.innerText;b.innerText='Copied';setTimeout(function(){{b.innerText=o;}},1200);"" title=""Copy"" style=""flex:0 0 auto;padding:2px 9px;font-size:11px;font-family:inherit;color:#1B5162;background:#F9F7F4;border:1px solid #EEEDE9;border-radius:5px;cursor:pointer"">Copy"
SQL function,"labuser17579966_1791942688.agent_training.avg_neigh_price<button onclick=""var t=document.createElement('textarea');t.value='labuser17579966_1791942688.agent_training.avg_neigh_price';document.body.appendChild(t);t.select();try{{document.execCommand('copy');}}catch(e){{}}document.body.removeChild(t);var b=this;var o=b.innerText;b.innerText='Copied';setTimeout(function(){{b.innerText=o;}},1200);"" title=""Copy"" style=""flex:0 0 auto;padding:2px 9px;font-size:11px;font-family:inherit;color:#1B5162;background:#F9F7F4;border:1px solid #EEEDE9;border-radius:5px;cursor:pointer"">Copy"
Model service,"labuser17579966_1791942688.agent_training.agent-training-model<button onclick=""var t=document.createElement('textarea');t.value='labuser17579966_1791942688.agent_training.agent-training-model';document.body.appendChild(t);t.select();try{{document.execCommand('copy');}}catch(e){{}}document.body.removeChild(t);var b=this;var o=b.innerText;b.innerText='Copied';setTimeout(function(){{b.innerText=o;}},1200);"" title=""Copy"" style=""flex:0 0 auto;padding:2px 9px;font-size:11px;font-family:inherit;color:#1B5162;background:#F9F7F4;border:1px solid #EEEDE9;border-radius:5px;cursor:pointer"">Copy"
MLflow experiment,"/Workspace/Users/labuser17579966_1791942688@vocareum.com/dspy-agent-training<button onclick=""var t=document.createElement('textarea');t.value='/Workspace/Users/labuser17579966_1791942688@vocareum.com/dspy-agent-training';document.body.appendChild(t);t.select();try{{document.execCommand('copy');}}catch(e){{}}document.body.removeChild(t);var b=this;var o=b.innerText;b.innerText='Copied';setTimeout(function(){{b.innerText=o;}},1200);"" title=""Copy"" style=""flex:0 0 auto;padding:2px 9px;font-size:11px;font-family:inherit;color:#1B5162;background:#F9F7F4;border:1px solid #EEEDE9;border-radius:5px;cursor:pointer"">Copy"


Turn on MLflow tracing for DSPy. Every DSPy call after this cell is traced, which you will use to see how DSPy builds prompts and calls tools.

In [0]:
import mlflow

mlflow.dspy.autolog()

### A1. Lab Scenario

<div class="scn8"><style>.scn8{font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,sans-serif;color:#0b2026;width:100%;max-width:100%;box-sizing:border-box;margin:0;padding:8px}.scn8 .scen{background:#FFFFFF;color:#0b2026;padding:26px 30px;border-radius:12px;box-sizing:border-box;overflow:hidden;box-shadow:0 2px 16px rgba(27,49,57,0.10),0 1px 4px rgba(27,49,57,0.06);border:1px solid rgba(27,49,57,0.08)}.scn8 .kicker{font-size:14px;text-transform:uppercase;letter-spacing:1.2px;color:#5A6F77;margin-bottom:10px;font-weight:700;display:flex;align-items:center;gap:10px}.scn8 .body{font-size:15px;line-height:1.7;color:#0b2026}.scn8 .body p{margin:0 0 14px}.scn8 .body p:last-child{margin:0}.scn8 .tools{background:#F9F7F4;border:1px solid rgba(27,49,57,0.10);border-radius:8px;padding:12px 16px;margin:0 0 14px 0;font-family:ui-monospace,SFMono-Regular,Consolas,monospace;font-size:14px;line-height:1.9}.scn8 .fn{color:#0B8F5F}.scn8 .arg{color:#5A6F77}.scn8 .dsc{color:#5A6F77;opacity:.8}.scn8 code{background:rgba(27,49,57,0.07);padding:2px 6px;border-radius:4px;font-size:14px;font-family:ui-monospace,SFMono-Regular,Consolas,monospace;color:#0b2026}.scn8 .eq{background:rgba(0,169,114,.10);border-left:4px solid #00A972;padding:10px 14px;border-radius:0 6px 6px 0;margin:10px 0 0;font-size:14px;font-style:italic;line-height:1.6;color:#0b2026}</style><div class="scen">
<div class="kicker"><svg width="20" height="20" viewBox="0 0 24 24" fill="none" stroke="#5A6F77" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><path d="M12 8V4H8"/><rect x="4" y="8" width="16" height="12" rx="2"/><path d="M2 14h2"/><path d="M20 14h2"/><path d="M9 13v2"/><path d="M15 13v2"/></svg>Scenario</div>
<div class="body">
<p>You are building an <strong>Airbnb recommendation assistant</strong> for San Francisco. A user should be able to ask about a specific listing, compare it against neighborhood averages, and get a concrete recommendation &#x2014; all in one turn. The assistant needs to <em>reason</em> about which tools to call and in what order, then synthesize the results into a single answer.</p>

<p>You will wire DSPy to a Databricks-hosted model, define two tools, and combine them in a <code>dspy.ReAct</code> agent that alternates between thinking and acting until it can answer:</p>

<div class="tools"><div><span class="fn">get_listing_details</span><span class="arg">(listing_id)</span> <span class="dsc">&#x2014; fetches structured details for one listing from a Spark table</span></div><div><span class="fn">avg_neigh_price</span><span class="arg">(neighborhood)</span> <span class="dsc">&#x2014; calls a Unity Catalog function that returns the neighborhood&#x2019;s average nightly price</span></div></div>

<p>The first tool is a plain Python function &#x2014; DSPy has no tool decorator, so any function can be a tool. The second wraps a UC function using a parameterized query, keeping the logic governed in Unity Catalog while DSPy sees an ordinary callable. By the end, the agent will handle questions like:</p>

<div class="eq">"Tell me about listing 958. Is it more expensive than the average listing in the Mission?"</div>
</div>
</div></div>

## B. Configure DSPy

### B1. Connect DSPy to a Model Service via Unity Gateway

The first step in getting started with DSPy is to create a language model client. `dspy.LM` creates the client, and there are two ways to bring in a model:
- `databricks/<llm_endpoint_name>`: calls a Databricks-provided model service from the `system.ai` catalog (available to all account users by default).
- `databricks/<catalog.schema.model_service>`: calls a custom model service registered elsewhere in Unity Catalog (with configurable routing, fallbacks, and access controls).

Authentication is handled via your Databricks credentials (in our case by defining a notebook-scoped token for demonstration purposes), and authorization is governed by Unity Catalog permissions after we have resolved the Workspace URL. `dspy.configure` then sets it as the default model for every DSPy module.

**TODO:** Fill out the fully-qualified name for bringing in the model service in your schema `agent_training` (inspect the name using the **Catalog** button to the left) and set the default model for all DSPy modules that will be used in this notebook.

In [0]:
import os
import dspy

token = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().apiToken().get()
)

host = spark.conf.get("spark.databricks.workspaceUrl")

# model_service = "<FILL_IN>"
model_service  = f"{catalog_name}.{schema_name}.agent-training-model"

lm = dspy.LM(
    model=f"databricks/{model_service}",  # fully qualified UC name
    api_base=f"https://{host}/ai-gateway/mlflow/v1",
    api_key=token
)

# dspy.configure(lm=<FILL_IN>)
dspy.configure(lm=lm)

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
import os
import dspy

token = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().apiToken().get()
)

host = spark.conf.get("spark.databricks.workspaceUrl")

model_service = f"{catalog_name}.{schema_name}.agent-training-model"

lm = dspy.LM(
    model=f"databricks/{model_service}",  # fully qualified UC name
    api_base=f"https://{host}/ai-gateway/mlflow/v1",
    api_key=token
)

dspy.configure(lm=lm)
```

</div>
</details>

## C. Signatures and Modules

A [**signature**](https://dspy.ai/current/api/signatures/) declares a task's inputs and outputs, and a [**module**](https://dspy.ai/current/api/modules/) decides how to prompt the model for it. The shorthand signature below has one input, `question`, and one output, `answer`. `dspy.Predict` is the simplest module.

You never write the prompt yourself. Inside every module, an [**adapter**](https://dspy.ai/current/api/adapters/) writes the prompt from your signature, sends it to the model, and parses the reply back into your output fields. The adapter pattern allows DSPy to work with different LM interfaces while maintaining a consistent programming model for users. The diagram follows one call through `dspy.Predict`:

<div class="dsp">
<style>
  .dsp{background:#ffffff;width:100%;max-width:none;margin:0;padding:8px 0 16px;box-sizing:border-box;}
  .dsp svg{display:block;width:100%;height:auto;}
  .dsp svg text{font-family:-apple-system,"Segoe UI",Roboto,Arial,sans-serif;fill:#0b2026;}
  .dsp .h   {font-weight:800;}
  .dsp .b   {font-weight:700;}
  .dsp .m   {fill:#618794;font-weight:500;}
  .dsp .mono,.dsp .mono text{white-space:pre;font-family:ui-monospace,Menlo,Consolas,monospace;font-variant-ligatures:none;font-feature-settings:"liga" 0,"calt" 0;}
</style>
<svg viewBox="0 0 1600 840" preserveAspectRatio="xMidYMid meet" role="img" aria-label="Inside dspy.Predict: the Adapter round trip, following one example. The Signature 'question -> answer' promises an answer for a question. 1, you call qa(question='What is the capital of France?'). 2, the Adapter's format step writes a system message listing the input and output fields and a user message with the question under a [[ ## question ## ]] marker. 3, the LM only sees those chat messages and replies with plain text. 4, the Adapter's parse step finds the [[ ## answer ## ]] marker in the reply and extracts Paris. 5, you get back Prediction(answer='Paris'). ChatAdapter is the default; JSONAdapter or TwoStepAdapter can be swapped in with dspy.configure(adapter=...) without changing the program.">
  <defs>
    <marker id="ar-b" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="6" markerHeight="6" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#4299E0"/></marker>
    <marker id="ar-g" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="6" markerHeight="6" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#00A972"/></marker>
    <marker id="ar-s" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="7" markerHeight="7" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#C4CCD6"/></marker>
    <filter id="sh" x="-10%" y="-10%" width="120%" height="130%"><feDropShadow dx="0" dy="4" stdDeviation="9" flood-color="#1B3139" flood-opacity="0.10"/></filter>
    <filter id="glow" x="-40%" y="-40%" width="180%" height="180%"><feDropShadow dx="0" dy="6" stdDeviation="12" flood-color="#1B5162" flood-opacity="0.30"/></filter>
  </defs>
  <!-- ================= the signature, drawn as a promise ================= -->
  <path d="M280.0,85 L1312.0,85" fill="none" stroke="#C4CCD6" stroke-width="2.5" marker-end="url(#ar-s)"/>
  <rect id="pill-q" x="92.0" y="62" width="184" height="46" rx="23" fill="#4299E0"/>
  <text class="mono b" x="184.0" y="92" font-size="19" text-anchor="middle" style="fill:#ffffff" data-box="pill-q">question</text>
  <rect id="pill-a" x="1324.0" y="62" width="184" height="46" rx="23" fill="#00A972"/>
  <text class="mono b" x="1416.0" y="92" font-size="19" text-anchor="middle" style="fill:#ffffff" data-box="pill-a">answer</text>
  <rect id="sig" x="560" y="66" width="480" height="38" rx="19" fill="#ffffff"/>
  <text x="800" y="92" font-size="19" text-anchor="middle" data-box="sig"><tspan class="b">Signature</tspan><tspan class="m">  </tspan><tspan class="mono m" font-size="17">"question -&gt; answer"</tspan></text>
  <!-- guides from the promise down to where it's kept -->
  <g fill="none" stroke="#C4CCD6" stroke-width="2" stroke-dasharray="6 6">
    <path d="M184.0,110 L184.0,318"/>
    <path d="M1416.0,110 L1416.0,329"/>
  </g>
  <!-- ================= the adapter arc: one object, both sides ================= -->
  <path d="M532.0,250 C532.0,184 612.0,171 800,171 C988.0,171 1068.0,184 1068.0,250" fill="none" stroke="#1B5162" stroke-width="3"/>
  <rect id="adp" x="684" y="148" width="232" height="46" rx="23" fill="#1B5162"/>
  <text x="800" y="178" font-size="21" text-anchor="middle" style="fill:#ffffff" data-box="adp" font-weight="800">Adapter</text>
  <text class="m" x="800" y="228" font-size="17" text-anchor="middle">ChatAdapter by default</text>
  <!-- ================= 1 · your call ================= -->
  <rect id="call" x="28" y="320" width="312" height="204" rx="12" fill="#303F47" filter="url(#sh)"/>
  <text x="50" y="354" font-size="16" style="fill:#90A5B1" data-box="call" font-weight="700" letter-spacing="1.4">YOUR CALL</text>
  <text class="mono" x="50" y="390" font-size="17" style="fill:#C4CCD6" data-box="call">qa = dspy.Predict(</text>
  <text class="mono" x="50" y="414" font-size="17" style="fill:#C4CCD6" data-box="call">  <tspan style="fill:#FFCC66">"question -&gt; answer"</tspan>)</text>
  <text class="mono" x="50" y="452" font-size="17" style="fill:#C4CCD6" data-box="call">qa(question=</text>
  <text class="mono" x="50" y="476" font-size="17" style="fill:#FFCC66" data-box="call">  "What is the capital</text>
  <text class="mono" x="50" y="500" font-size="17" style="fill:#C4CCD6" data-box="call"><tspan style="fill:#FFCC66">   of France?"</tspan>)</text>
  <path d="M344,422 L368,422" fill="none" stroke="#4299E0" stroke-width="2.5" marker-end="url(#ar-b)"/>
  <!-- ================= 2 · format() ================= -->
  <path id="fmt" d="M376,250 H678 A10,10 0 0 1 688,260 V584 A10,10 0 0 1 678,594 H376 Z" fill="#F9F7F4" stroke="#EEEDE9" filter="url(#sh)"/>
  <rect x="376" y="250" width="5" height="344" fill="#1B5162"/>
  <text class="mono b" x="396" y="292" font-size="26" data-box="fmt">format()</text>
  <text class="m" x="396" y="318" font-size="17" data-box="fmt">signature + demos + inputs</text>
  <rect id="sys" x="396" y="332" width="278" height="128" rx="8" fill="#ffffff" stroke="#DCE0E2"/>
  <text x="410" y="356" font-size="16" style="fill:#618794" data-box="sys" font-weight="700" letter-spacing="1">SYSTEM</text>
  <text class="mono" x="410" y="380" font-size="17" data-box="sys">Your input fields are:</text>
  <text class="mono" x="410" y="402" font-size="17" data-box="sys">1. <tspan style="fill:#4299E0">`question`</tspan> (str)</text>
  <text class="mono" x="410" y="424" font-size="17" data-box="sys">Your output fields are:</text>
  <text class="mono" x="410" y="446" font-size="17" data-box="sys">1. <tspan style="fill:#00A972">`answer`</tspan> (str) …</text>
  <rect id="usr" x="396" y="472" width="278" height="106" rx="8" fill="#ffffff" stroke="#DCE0E2"/>
  <text x="410" y="496" font-size="16" style="fill:#618794" data-box="usr" font-weight="700" letter-spacing="1">USER</text>
  <text class="mono" x="410" y="520" font-size="17" style="fill:#4299E0" data-box="usr">[[ ## question ## ]]</text>
  <text class="mono" x="410" y="542" font-size="17" data-box="usr">What is the capital of</text>
  <text class="mono m" x="410" y="564" font-size="17" data-box="usr">France? Respond with …</text>
  <path d="M692,422 L728,422" fill="none" stroke="#4299E0" stroke-width="2.5" marker-end="url(#ar-b)"/>
  <!-- ================= 3 · the LM ================= -->
  <text class="mono b" x="800" y="334" font-size="19" text-anchor="middle">LM.__call__</text>
  <circle cx="800" cy="422" r="64" fill="#1B5162" filter="url(#glow)"/>
  <g stroke="#ffffff" stroke-width="3.2" fill="#ffffff">
    <line x1="782" y1="407" x2="816" y2="401"/>
    <line x1="782" y1="407" x2="820" y2="438"/>
    <line x1="816" y1="401" x2="820" y2="438"/>
    <line x1="782" y1="407" x2="786" y2="442"/>
    <line x1="820" y1="438" x2="786" y2="442"/>
    <circle cx="782" cy="407" r="6"/>
    <circle cx="816" cy="401" r="6"/>
    <circle cx="820" cy="438" r="6"/>
    <circle cx="786" cy="442" r="6"/>
  </g>
  <text class="m" x="800" y="520" font-size="17" text-anchor="middle">chat messages in,</text>
  <text class="m" x="800" y="543" font-size="17" text-anchor="middle">plain text out</text>
  <path d="M868,422 L904,422" fill="none" stroke="#00A972" stroke-width="2.5" marker-end="url(#ar-g)"/>
  <!-- ================= 4 · parse() ================= -->
  <path id="prs" d="M912,250 H1214 A10,10 0 0 1 1224,260 V584 A10,10 0 0 1 1214,594 H912 Z" fill="#F9F7F4" stroke="#EEEDE9" filter="url(#sh)"/>
  <rect x="912" y="250" width="5" height="344" fill="#1B5162"/>
  <!-- adapter stem-end dots: centered on the card top edge so the border bisects each circle -->
  <circle cx="532.0" cy="250" r="6" fill="#1B5162"/>
  <circle cx="1068.0" cy="250" r="6" fill="#1B5162"/>
  <text class="mono b" x="932" y="292" font-size="26" data-box="prs">parse()</text>
  <text class="m" x="932" y="318" font-size="17" data-box="prs">reply → typed output fields</text>
  <rect id="rep" x="932" y="332" width="278" height="126" rx="8" fill="#ffffff" stroke="#DCE0E2"/>
  <text x="946" y="356" font-size="16" style="fill:#618794" data-box="rep" font-weight="700" letter-spacing="1">LM REPLY</text>
  <text class="mono" x="946" y="382" font-size="17" style="fill:#00A972" data-box="rep">[[ ## answer ## ]]</text>
  <rect id="paris" x="940" y="389" width="72" height="35" rx="6" fill="rgba(0,169,114,0.14)"/>
  <text class="mono b" x="946" y="412" font-size="17" data-box="paris">Paris</text>
  <text class="mono m" x="946" y="444" font-size="17" data-box="rep">[[ ## completed ## ]]</text>
  <rect id="ans" x="932" y="474" width="278" height="46" rx="8" fill="#00A972"/>
  <text class="mono b" x="1071.0" y="503" font-size="17" text-anchor="middle" style="fill:#ffffff" data-box="ans">answer = "Paris"</text>
  <text class="m" x="1071.0" y="556" font-size="17" text-anchor="middle" data-box="prs">checked against the output type</text>
  <path d="M1228,422 L1252,422" fill="none" stroke="#00A972" stroke-width="2.5" marker-end="url(#ar-g)"/>
  <!-- ================= 5 · what you get back ================= -->
  <path id="res" d="M1260,331 H1562 A10,10 0 0 1 1572,341 V503 A10,10 0 0 1 1562,513 H1260 Z" fill="#ffffff" stroke="#DCE0E2" stroke-width="1.5" filter="url(#sh)"/>
  <rect x="1260" y="331" width="5" height="182" fill="#00A972"/>
  <text x="1282" y="365" font-size="16" style="fill:#00A972" data-box="res" font-weight="700" letter-spacing="1.4">YOU GET BACK</text>
  <text class="mono" x="1282" y="403" font-size="17" data-box="res">Prediction(</text>
  <text class="mono" x="1282" y="429" font-size="17" data-box="res">  answer=<tspan class="b" style="fill:#00A972">"Paris"</tspan>)</text>
  <line x1="1282" y1="453" x2="1550" y2="453" stroke="#EEEDE9" stroke-width="1.5"/>
  <text class="mono m" x="1282" y="485" font-size="17" data-box="res">pred.answer → "Paris"</text>
  <!-- ================= step captions ================= -->
  <g font-size="18" text-anchor="middle">
    <circle cx="184.0" cy="648" r="16" fill="#4299E0"/><text x="184.0" y="654" font-size="18" font-weight="700" style="fill:#ffffff">1</text>
    <text class="m" x="184.0" y="692" data-cap="1">Call with keyword args</text>
    <text class="m" x="184.0" y="716" data-cap="1">named by the signature</text>
    <circle cx="532.0" cy="648" r="16" fill="#1B5162"/><text x="532.0" y="654" font-size="18" font-weight="700" style="fill:#ffffff">2</text>
    <text class="m" x="532.0" y="692" data-cap="2">format() writes the prompt,</text>
    <text class="m" x="532.0" y="716" data-cap="2">marking each field by name</text>
    <circle cx="800" cy="648" r="16" fill="#1B5162"/><text x="800" y="654" font-size="18" font-weight="700" style="fill:#ffffff">3</text>
    <text class="m" x="800" y="692" data-cap="3">The LM never sees</text>
    <text class="m" x="800" y="716" data-cap="3">your Python types</text>
    <circle cx="1068.0" cy="648" r="16" fill="#1B5162"/><text x="1068.0" y="654" font-size="18" font-weight="700" style="fill:#ffffff">4</text>
    <text class="m" x="1068.0" y="692" data-cap="4">parse() finds the markers</text>
    <text class="m" x="1068.0" y="716" data-cap="4">and pulls out each field</text>
    <circle cx="1416.0" cy="648" r="16" fill="#00A972"/><text x="1416.0" y="654" font-size="18" font-weight="700" style="fill:#ffffff">5</text>
    <text class="m" x="1416.0" y="692" data-cap="5">Typed fields come back</text>
    <text class="m" x="1416.0" y="716" data-cap="5">as a Prediction</text>
  </g>
  <!-- ================= swap strip ================= -->
  <line x1="28" y1="748" x2="1572" y2="748" stroke="#EEEDE9" stroke-width="1.5"/>
  <text class="b" x="28" y="800" font-size="20" data-row="swap">Swap the adapter, keep the program</text>
  <g font-size="17">
    <rect id="p1" x="412" y="772" width="248" height="44" rx="22" fill="#1B5162" data-row="swap"/>
    <text x="536" y="800" text-anchor="middle" style="fill:#ffffff" data-box="p1"><tspan class="mono b">ChatAdapter</tspan> · default</text>
    <rect id="p2" x="672" y="772" width="160" height="44" rx="22" fill="#ffffff" stroke="#1B5162" stroke-width="1.5" data-row="swap"/>
    <text class="mono b" x="752" y="800" text-anchor="middle" data-box="p2">JSONAdapter</text>
    <rect id="p3" x="844" y="772" width="190" height="44" rx="22" fill="#ffffff" stroke="#1B5162" stroke-width="1.5" data-row="swap"/>
    <text class="mono b" x="939" y="800" text-anchor="middle" data-box="p3">TwoStepAdapter</text>
  </g>
  <text class="mono m" x="1572" y="800" font-size="17" text-anchor="end" data-row="swap">dspy.configure(adapter=JSONAdapter())</text>
</svg>
</div>

Run the next cell, then open the trace below the output. You will see the same round trip: `ChatAdapter.format`, then `LM.__call__`, then the parsed `answer`.

In [0]:
signature = "question -> answer: str"
question = "4 dice are tossed. What is the probability that the sum of the dice equals 2 or 4?"

qa = dspy.Predict(signature)
qa(question=question)

Prediction(
    answer='With 4 standard six-sided dice, the smallest possible sum is \\(1+1+1+1=4\\), so a sum of 2 is impossible (probability 0).\n\nA sum of 4 occurs only if all four dice show 1, i.e., \\((1,1,1,1)\\), which is 1 outcome out of \\(6^4=1296\\) total equally likely outcomes.\n\n\\[\nP(\\text{sum}=2\\text{ or }4)=P(\\text{sum}=4)=\\frac{1}{6^4}=\\frac{1}{1296}.\n\\]'
)

Trace(trace_id=tr-f3223a5cc97dd73de04aa4bc02ed91dd)

**TODO:** Run the same `signature` through `dspy.ChainOfThought`, which asks the model to reason step by step before answering. Compare the two traces: Chain of Thought adds a `reasoning` output field and an extra `Predict.forward` step, but the adapter round trip is the same.

In [0]:
# qa_cot = dspy.<FILL_IN>(<FILL_IN>)
qa_cot = dspy.ChainOfThought(signature)
qa_cot(question=question)

Prediction(
    reasoning='With 4 standard six-sided dice, the smallest possible sum is \\(1+1+1+1=4\\), so getting a sum of 2 is impossible (probability 0).  \nA sum of 4 occurs only when all four dice show 1, which is 1 outcome out of \\(6^4\\) equally likely outcomes. Thus\n\\[\nP(\\text{sum }=2 \\text{ or } 4)=P(\\text{sum }=4)=\\frac{1}{6^4}=\\frac{1}{1296}.\n\\]',
    answer='\\[\n\\boxed{\\frac{1}{1296}}\n\\]'
)

Trace(trace_id=tr-974ed0db0461d1b60a1b290dfd3e6e32)

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
qa_cot = dspy.ChainOfThought(signature)
qa_cot(question=question)
```

</div>
</details>

## D. Prepare the Data

We will create an agent that answers questions about the San Francisco Airbnb listings created by our classroom setup script. Run the next cell to preview the table.

In [0]:
%sql
SELECT * FROM sf_airbnb_listings LIMIT 5

id,name,neighbourhood,neighbourhood_cleansed,summary,price,room_type,listing_source_information
958,"Bright, Modern Garden Unit - 1BR/1B",Duboce Triangle,Western Addition,New update: the house next door is under construction and there is the possibility of noise from 7am - 5pm. Our rates are discounted during this time period. Our bright garden unit overlooks a grassy backyard area with fruit trees and native plants. It is an oasis in a big city. The apartment comfortably fits a couple or small family. It is located on a cul de sac street that ends at lovely Duboce Park.,170.0,Entire home/apt,"ID of the property: 958 Name of the property: Bright, Modern Garden Unit - 1BR/1B Summary of the property: New update: the house next door is under construction and there is the possibility of noise from 7am - 5pm. Our rates are discounted during this time period. Our bright garden unit overlooks a grassy backyard area with fruit trees and native plants. It is an oasis in a big city. The apartment comfortably fits a couple or small family. It is located on a cul de sac street that ends at lovely Duboce Park."
5858,Creative Sanctuary,Bernal Heights,Bernal Heights,null,235.0,Entire home/apt,null
7918,A Friendly Room - UCSF/USF - San Francisco,Cole Valley,Haight Ashbury,"Nice and good public transportation. 7 minutes walk to UCSF. 15 minutes walk to USF, ST Mary Hospital Room rental-sunny view room/sink/Wi Fi (inner sunset / UCSF) (map) The place is located within walking distance to Golden Gate Park, UCSF & USF",65.0,Private room,"ID of the property: 7918 Name of the property: A Friendly Room - UCSF/USF - San Francisco Summary of the property: Nice and good public transportation. 7 minutes walk to UCSF. 15 minutes walk to USF, ST Mary Hospital Room rental-sunny view room/sink/Wi Fi (inner sunset / UCSF) (map) The place is located within walking distance to Golden Gate Park, UCSF & USF"
8142,Friendly Room Apt. Style -UCSF/USF - San Francisco,Cole Valley,Haight Ashbury,"Nice and good public transportation. 7 minutes walk to UCSF. 15 minutes walk to USF, ST Mary Hospital Room rental-sunny view room/sink/Wi Fi (inner sunset / UCSF) (map) The place is located within walking distance to Golden Gate Park, UCSF & USF",65.0,Private room,"ID of the property: 8142 Name of the property: Friendly Room Apt. Style -UCSF/USF - San Francisco Summary of the property: Nice and good public transportation. 7 minutes walk to UCSF. 15 minutes walk to USF, ST Mary Hospital Room rental-sunny view room/sink/Wi Fi (inner sunset / UCSF) (map) The place is located within walking distance to Golden Gate Park, UCSF & USF"
8339,Historic Alamo Square Victorian,Western Addition/NOPA,Western Addition,"Pls email before booking. Interior featured in magazines. Listed historic with the City of SF. Steps away from the iconic Alamo Square ""Painted Ladies"". European feel, reminiscent of the Boutique Hotels in Paris, Berlin, Antwerp and London.",785.0,Entire home/apt,"ID of the property: 8339 Name of the property: Historic Alamo Square Victorian Summary of the property: Pls email before booking. Interior featured in magazines. Listed historic with the City of SF. Steps away from the iconic Alamo Square ""Painted Ladies"". European feel, reminiscent of the Boutique Hotels in Paris, Berlin, Antwerp and London."


The next cell defines a [Pydantic model](https://pydantic.dev/), `ListingInfo`, that describes a listing. Every field on a Signature is a Pydantic `FieldInfo` and the base `Signature` class extends `pydantic.BaseModel`. Your first tool returns it, so the agent receives structured, typed fields instead of a raw row.

In [0]:
from typing import Optional
from pydantic import BaseModel

class ListingInfo(BaseModel):
    """Key details about an Airbnb listing."""
    id: int
    name: str
    neighbourhood: str
    neighbourhood_cleansed: str
    summary: Optional[str]
    price: float
    room_type: str
    listing_source_information: Optional[str]

## E. Define the Agent's Tools

### E1. Prototype a Tool as a Plain Function

DSPy has no tool decorator: any Python function can be a tool. Instead, we can pass the tool directly to the agent without any additional configuration.

**TODO:** Finish `get_listing_details` by adding:
- a type hint for `listing_id` (an integer) and the return type (`ListingInfo`)
- a docstring that says what the tool returns and when to use it

The body is provided.

In [0]:
# def get_listing_details(listing_id: <FILL_IN>) -> <FILL_IN>:
#     """<FILL_IN>"""
def get_listing_details(listing_id: int) -> ListingInfo:
    """Get details for one San Francisco Airbnb listing by its ID, including its
    neighborhood, room type, nightly price in USD, and summary.

    Use this when the user asks about a specific listing ID."""
    rows = spark.table("sf_airbnb_listings").filter(f"id = {int(listing_id)}").limit(1).collect()
    if not rows:
        raise ValueError(f"Listing with ID {listing_id} not found.")
    r = rows[0]
    return ListingInfo(
        id=r.id,
        name=r.name,
        neighbourhood=r.neighbourhood,
        neighbourhood_cleansed=r.neighbourhood_cleansed,
        summary=r.summary,
        price=r.price,
        room_type=r.room_type,
        listing_source_information=r.listing_source_information,
    )

get_listing_details(958)

ListingInfo(id=958, name='Bright, Modern Garden Unit - 1BR/1B', neighbourhood='Duboce Triangle', neighbourhood_cleansed='Western Addition', summary='New update: the house next door is under construction and there is the possibility of noise from 7am - 5pm. Our rates are discounted during this time period. Our bright garden unit overlooks a grassy backyard area with fruit trees and native plants. It is an oasis in a big city. The apartment comfortably fits a couple or small family. It is located on a cul de sac street that ends at lovely Duboce Park.', price=170.0, room_type='Entire home/apt', listing_source_information='ID of the property: 958\nName of the property: Bright, Modern Garden Unit - 1BR/1B\nSummary of the property: New update: the house next door is under construction and there is the possibility of noise from 7am - 5pm. Our rates are discounted during this time period. Our bright garden unit overlooks a grassy backyard area with fruit trees and native plants. It is an oasi

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
def get_listing_details(listing_id: int) -> ListingInfo:
    """Get details for one San Francisco Airbnb listing by its ID, including its
    neighborhood, room type, nightly price in USD, and summary.

    Use this when the user asks about a specific listing ID."""
    rows = spark.table("sf_airbnb_listings").filter(f"id = {int(listing_id)}").limit(1).collect()
    if not rows:
        raise ValueError(f"Listing with ID {listing_id} not found.")
    r = rows[0]
    return ListingInfo(
        id=r.id,
        name=r.name,
        neighbourhood=r.neighbourhood,
        neighbourhood_cleansed=r.neighbourhood_cleansed,
        summary=r.summary,
        price=r.price,
        room_type=r.room_type,
        listing_source_information=r.listing_source_information,
    )

get_listing_details(958)
```

</div>
</details>

### E2. Wrap a Unity Catalog Function as a Tool

Setup registered `avg_neigh_price` to Unity Catalog. To give it to a DSPy agent, wrap a call to it in a plain function. The logic stays governed in Unity Catalog, and DSPy sees an ordinary tool.

The neighborhood comes from the model, so never paste it into the SQL text. Use a **named parameter marker** such as `:neighborhood` and pass the value in `args`. Spark then treats it as data, so a value with a quote in it cannot break, or change, the query.

**TODO:** Fill in the parameter marker and the `args` dictionary.

In [0]:
def avg_neigh_price(neighborhood: str) -> Optional[float]:
    """Get the average nightly price in USD of San Francisco Airbnb listings in a
    neighborhood, such as 'Mission' or 'Noe Valley'."""
    # result = spark.sql(
    #     "SELECT avg_neigh_price(<FILL_IN>) AS average_price",
    #     args=<FILL_IN>,
    # ).collect()
    result = spark.sql(
        "SELECT avg_neigh_price(:neighborhood) AS average_price",
        args={"neighborhood": neighborhood},
    ).collect()
    return result[0]["average_price"] if result else None

avg_neigh_price("Mission")

188.625

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
def avg_neigh_price(neighborhood: str) -> Optional[float]:
    """Get the average nightly price in USD of San Francisco Airbnb listings in a
    neighborhood, such as 'Mission' or 'Noe Valley'."""
    result = spark.sql(
        "SELECT avg_neigh_price(:neighborhood) AS average_price",
        args={"neighborhood": neighborhood},
    ).collect()
    return result[0]["average_price"] if result else None

avg_neigh_price("Mission")
```

</div>
</details>

## F. Build the ReAct Agent

### F1. Define the Agent's Signature

For an agent, write the signature as a class. The docstring becomes the agent's instructions, and each field gets a description the model reads.

**TODO:** Declare `user_request` as an input field and `response` as an output field.

In [0]:
class AirbnbRecommendationAgent(dspy.Signature):
    """You are an Airbnb recommendation assistant for San Francisco listings.
    Use your tools to look up listings and neighborhood prices, and answer with specific numbers."""

    # user_request: str = dspy.<FILL_IN>(desc="The user's question about Airbnb listings")
    # response: str = dspy.<FILL_IN>(desc="A helpful answer with specific listing details and prices")
    user_request: str = dspy.InputField(desc="The user's question about Airbnb listings")
    response: str = dspy.OutputField(desc="A helpful answer with specific listing details and prices")

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
class AirbnbRecommendationAgent(dspy.Signature):
    """You are an Airbnb recommendation assistant for San Francisco listings.
    Use your tools to look up listings and neighborhood prices, and answer with specific numbers."""

    user_request: str = dspy.InputField(desc="The user's question about Airbnb listings")
    response: str = dspy.OutputField(desc="A helpful answer with specific listing details and prices")
```

</div>
</details>

### F2. Create the Agent

`dspy.ReAct` alternates between **reasoning** about what to do next and **acting** by calling a tool, until it can answer.

**TODO:** Create the agent from your signature and both tools. The output lists three tools: your two, plus `finish`, which ReAct always adds so the agent can signal that it's done.

In [0]:
# agent = dspy.ReAct(
#     <FILL_IN>,
#     tools=[<FILL_IN>, <FILL_IN>],
# )

agent = dspy.ReAct(
    AirbnbRecommendationAgent,
    tools=[get_listing_details, avg_neigh_price],
)

for name, tool in agent.tools.items():
    print(f"- {name}: {tool.desc[:90]}")

- get_listing_details: Get details for one San Francisco Airbnb listing by its ID, including its
    neighborhood
- avg_neigh_price: Get the average nightly price in USD of San Francisco Airbnb listings in a
    neighborhoo
- finish: Marks the task as complete. That is, signals that all information for producing the output


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
agent = dspy.ReAct(
    AirbnbRecommendationAgent,
    tools=[get_listing_details, avg_neigh_price],
)

for name, tool in agent.tools.items():
    print(f"- {name}: {tool.desc[:90]}")
```

</div>
</details>

## G. Run and Inspect the Agent

### G1. Ask a Question That Needs Both Tools

**TODO:** Call the agent, passing the question as the signature's input field, and print the signature's output field. Then open the trace under the output: you should see a call to `get_listing_details` and a call to `avg_neigh_price`.

In [0]:
# result = agent(<FILL_IN>="Tell me about listing 958. Is it more expensive than the average listing in the Mission?")

# print(result.<FILL_IN>)
result = agent(user_request="Tell me about listing 958. Is it more expensive than the average listing in the Mission?")

print(result.response)

**Listing 958: “Bright, Modern Garden Unit - 1BR/1B”**  
- **Location:** Duboce Triangle (neighbourhood_cleansed: Western Addition)  
- **Room type:** Entire home/apt  
- **Price:** **$170/night**

**Mission average nightly price:** **$188.63/night**

**Comparison:** No—**listing 958 is cheaper than the average Mission listing** by about **$18.63/night**.


Trace(trace_id=tr-b654cd8f14f6eb973bcff85cea2b6a8a)

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none; display: flex; align-items: center; gap: 10px;">
<span style="background: rgba(255,255,255,0.2); border-radius: 4px; padding: 2px 8px; font-size: 11pt;">ANSWER</span> Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
result = agent(user_request="Tell me about listing 958. Is it more expensive than the average listing in the Mission?")

print(result.response)
```

</div>
</details>

### G2. Inspect the Trajectory

Besides the answer, the result holds the agent's **trajectory**: for each step, its thought, the tool it chose, the arguments, and the result it observed. Run the next cell to print it.

In [0]:
for key, value in result.trajectory.items():
    print(f"{key}: {str(value)[:200]}")

thought_0: Need listing 958 details (price, neighborhood) and Mission average price to compare.
tool_name_0: get_listing_details
tool_args_0: {'listing_id': 958}
observation_0: id=958 name='Bright, Modern Garden Unit - 1BR/1B' neighbourhood='Duboce Triangle' neighbourhood_cleansed='Western Addition' summary='New update: the house next door is under construction and there is 
thought_1: I have listing 958’s nightly price ($170) and details; now I need the Mission neighborhood average nightly price to compare.
tool_name_1: avg_neigh_price
tool_args_1: {'neighborhood': 'Mission'}
observation_1: 188.625
thought_2: We have listing 958 details and Mission average price; now we can answer and finish.
tool_name_2: finish
tool_args_2: {}
observation_2: Completed.


To see the exact prompts DSPy sent to the model, use `dspy.inspect_history`.

In [0]:
dspy.inspect_history(n=2)





[2026-10-14T04:14:15.039843]

System message:

Your input fields are:
1. `user_request` (str): The user's question about Airbnb listings
2. `trajectory` (str):
Your output fields are:
1. `next_thought` (str): 
2. `next_tool_name` (Literal['get_listing_details', 'avg_neigh_price', 'finish']): 
3. `next_tool_args` (dict[str, Any]):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## user_request ## ]]
{user_request}

[[ ## trajectory ## ]]
{trajectory}

[[ ## next_thought ## ]]
{next_thought}

[[ ## next_tool_name ## ]]
{next_tool_name}        # note: the value you produce must exactly match (no extra characters) one of: get_listing_details; avg_neigh_price; finish

[[ ## next_tool_args ## ]]
{next_tool_args}        # note: the value you produce must adhere to the JSON schema: {"type": "object", "additionalProperties": true}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        You are an Airbnb recommend

## Conclusion

In this lab, you:
1. Connected DSPy to a Databricks model with `dspy.LM` and `dspy.configure`
2. Declared tasks with shorthand and class-based signatures, and let modules write the prompts
3. Prototyped a tool as a plain Python function and wrapped a Unity Catalog function as a second tool, using a parameterized query
4. Built a `dspy.ReAct` agent and inspected its trajectory and MLflow trace

DSPy's biggest strength, **optimizing** prompts and modules against your own data, builds on exactly these signatures and modules. To go further, see the [DSPy documentation](https://dspy.ai/) and the Databricks guide to DSPy (<a href="https://docs.databricks.com/aws/en/generative-ai/dspy/" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/generative-ai/dspy/" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/generative-ai/dspy/" style="color: #1976d2; text-decoration: underline;">GCP</a>).

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>